# [LELEC2870] - Machine Learning
## Part 3 — Integration of Image Data

**Authors:** ton_nom, mate_nom  
**Date:** 2025-12-01

In this notebook, we integrate heart scan images into the predictive pipeline. We extract features from images using a CNN and combine them with tabular data for prediction.


In [ ]:
# Standard libraries
import os
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

# Image processing
from PIL import Image

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset

# Sklearn
from sklearn.model_selection import train_test_split

# Utils from the practical session (adapt if needed)
from P5_utils import (
    visualize_sample_images, 
    visualize_dataset_tSNE,
    visualize_2Dconvolution,
    visualize_regression_results
)

import warnings
warnings.filterwarnings('ignore')


### 2.1 Load Tabular and Image Data


In [ ]:
# Load image filenames
images = pd.read_csv("data/X_train.csv")["img_filename"]

# Load tabular targets
y = pd.read_csv("data/y_train.csv", header=None, names=['risk']).values

# Optional: load other tabular features
tabular_data = pd.read_csv("data/X_tabular.csv")  # exemple


### 2.2 Define Custom Dataset


In [ ]:
class CustomDataset(Dataset):
    def __init__(self, images, images_directory, targets=None, transform=None):
        self.images = images
        self.images_directory = images_directory
        self.targets = targets
        self.transform = transform or transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5])
        ])
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = os.path.join(self.images_directory, self.images[idx])
        img = Image.open(img_path)
        if self.transform:
            img = self.transform(img)
        if self.targets is not None:
            return img, self.targets[idx]
        return img


In [ ]:
# Display sample images
dataset = CustomDataset(images, "data/Img_train", y)
visualize_sample_images(dataset, gridsize=(3,6))

# Optional: t-SNE on raw images
visualize_dataset_tSNE(dataset)


In [ ]:
# Example CNN class (reuse SimpleCNN from practical session)
class SimpleCNN(nn.Module):
    def __init__(self, n_features):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 8, 3, stride=1, padding=1)
        self.pool1 = nn.MaxPool2d(2,2)
        self.conv2 = nn.Conv2d(8, 8, 3, stride=1, padding=1)
        self.pool2 = nn.MaxPool2d(2,2)
        self.fc1 = nn.Linear(8*12*12, n_features)
        self.fc2 = nn.Linear(n_features, 1)
    
    def forward(self, x):
        x = self.pool1(torch.relu(self.conv1(x)))
        x = self.pool2(torch.relu(self.conv2(x)))
        x = x.view(-1, 8*12*12)
        features = self.fc1(x)
        out = self.fc2(features)
        return out, features


In [ ]:
# Train/validation split
images_train, images_val, y_train, y_val = train_test_split(images, y, test_size=0.2, random_state=42)

# Dataset and dataloader
train_dataset = CustomDataset(images_train, "data/Img_train", y_train)
val_dataset = CustomDataset(images_val, "data/Img_train", y_val)

train_loader = DataLoader(train_dataset, batch_size=50, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=50, shuffle=False)

# Instantiate model, criterion, optimizer
cnn = SimpleCNN(n_features=8)
criterion = nn.MSELoss()
optimizer = optim.Adam(cnn.parameters(), lr=0.0005)

# Training loop (simplified)
n_epochs = 20
for epoch in range(n_epochs):
    cnn.train()
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs, _ = cnn(inputs)
        loss = criterion(outputs.squeeze(), labels.float())
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}/{n_epochs} completed")


In [ ]:
# Extract 8D features from CNN for all images
dataset = CustomDataset(images, "data/Img_train", y)
all_features = []
cnn.eval()
with torch.no_grad():
    for img, _ in DataLoader(dataset, batch_size=50):
        _, feats = cnn(img)
        all_features.append(feats)
all_features = torch.cat(all_features).numpy()


In [ ]:
# Example: concatenate image features with tabular data
X_combined = np.hstack([tabular_data.values, all_features])
y_combined = y


In [ ]:
# Use best nonlinear model from Part 2 (e.g., RandomForest, MLP)
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_combined, y_combined)

# Predictions
y_pred = model.predict(X_combined)

# Performance visualization
visualize_regression_results(y_combined, y_pred)


- Compare performance with and without image features  
- Discuss importance of image-derived features  
- Observations on t-SNE embedding of CNN features vs raw pixels  
